In [ ]:
#NOTE: 
#model_v1 and model_v2 are not the final model, they are just retrained model to get the final model "model_v3".
#locate "training V3", as that is the final model used for this project, i had to train the model 3 times.
#Also locate below "final testing with model_v3" to see the testing output.  

In [ ]:
# Install required libraries
!pip install --upgrade pip
!pip install ultralytics opencv-python numpy matplotlib

In [ ]:
# Verify installation
from ultralytics import YOLO
import cv2#
import numpy as np
import matplotlib.pyplot as plt

print("Ultralytics YOLO installed successfully!")
print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)
print("Matplotlib is available.")

# Segmentation and creating Yaml for image preprossesing 

In [ ]:
import cv2
import numpy as np
import os
import shutil
import random

# Define paths
image_dir = "/Users/deen/Desktop/head count/images/"  # Directory with full images and annotations
segmented_dir = "segmented_images"  # Directory to save segmented images and annotations
dataset_dir = "head_count_dataset1"  # Directory for the final dataset
train_dir = os.path.join(dataset_dir, "train")
val_dir = os.path.join(dataset_dir, "val")
train_images_dir = os.path.join(train_dir, "images")
train_labels_dir = os.path.join(train_dir, "labels")
val_images_dir = os.path.join(val_dir, "images")
val_labels_dir = os.path.join(val_dir, "labels")

# Create directories
os.makedirs(segmented_dir, exist_ok=True)
for directory in [train_images_dir, train_labels_dir, val_images_dir, val_labels_dir]:
    os.makedirs(directory, exist_ok=True)

# List of image-annotation pairs
image_annotation_pairs = [
    ("/Users/deen/Desktop/head count/images/DSC09698.JPG", "/Users/deen/Desktop/head count/images/DSC09698.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09715.JPG", "/Users/deen/Desktop/head count/images/DSC09715.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09716.JPG", "/Users/deen/Desktop/head count/images/DSC09716.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09717.JPG", "/Users/deen/Desktop/head count/images/DSC09717.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09718.JPG", "/Users/deen/Desktop/head count/images/DSC09718.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09719.JPG", "/Users/deen/Desktop/head count/images/DSC09719.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09720.JPG", "/Users/deen/Desktop/head count/images/DSC09720.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09721.JPG", "/Users/deen/Desktop/head count/images/DSC09721.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09722.JPG", "/Users/deen/Desktop/head count/images/DSC09722.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09723.JPG", "/Users/deen/Desktop/head count/images/DSC09723.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09724.JPG", "/Users/deen/Desktop/head count/images/DSC09724.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09725.JPG", "/Users/deen/Desktop/head count/images/DSC09725.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09726.JPG", "/Users/deen/Desktop/head count/images/DSC09726.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09727.JPG", "/Users/deen/Desktop/head count/images/DSC09727.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09728.JPG", "/Users/deen/Desktop/head count/images/DSC09728.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09729.JPG", "/Users/deen/Desktop/head count/images/DSC09729.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09730.JPG", "/Users/deen/Desktop/head count/images/DSC09730.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09731.JPG", "/Users/deen/Desktop/head count/images/DSC09731.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09732.JPG", "/Users/deen/Desktop/head count/images/DSC09732.txt"),
    ("/Users/deen/Desktop/head count/images/DSC09733.JPG", "/Users/deen/Desktop/head count/images/DSC09733.txt"),
]

# Function to segment an image and its annotations
def segment_image(image_path, annotation_path, output_dir):
    # Load the image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Could not load image at {image_path}")
        return
    
    height, width = img.shape[:2]
    image_name = os.path.splitext(os.path.basename(image_path))[0]

    # Load annotations
    annotations = []
    if os.path.exists(annotation_path):
        with open(annotation_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    print(f"Invalid annotation format in {annotation_path}: {line}")
                    continue
                class_id, x_center, y_center, w, h = map(float, parts)
                x_center *= width
                y_center *= height
                w *= width
                h *= height
                annotations.append((class_id, x_center, y_center, w, h))
    else:
        print(f"Annotation file not found at {annotation_path}")
        return

    # Segment the image into 4 quadrants
    segments = [
        img[0:height//2, 0:width//2],          # Top-left
        img[0:height//2, width//2:width],      # Top-right
        img[height//2:height, 0:width//2],     # Bottom-left
        img[height//2:height, width//2:width]  # Bottom-right
    ]
    segment_names = ["top_left", "top_right", "bottom_left", "bottom_right"]
    segment_dims = [
        (0, 0, width//2, height//2),
        (width//2, 0, width, height//2),
        (0, height//2, width//2, height),
        (width//2, height//2, width, height)
    ]

    # Process each segment
    for i, (segment, name, (x_min, y_min, x_max, y_max)) in enumerate(zip(segments, segment_names, segment_dims)):
        segment_annotations = []
        
        # Filter annotations for this segment
        for class_id, x_center, y_center, w, h in annotations:
            x1 = x_center - w/2
            y1 = y_center - h/2
            x2 = x_center + w/2
            y2 = y_center + h/2
            
            # Check if the bounding box intersects this segment
            if (x1 < x_max and x2 > x_min and y1 < y_max and y2 > y_min):
                # Adjust coordinates relative to segment
                new_x_center = (max(x_min, x1) + min(x_max, x2)) / 2 - x_min
                new_y_center = (max(y_min, y1) + min(y_max, y2)) / 2 - y_min
                new_w = min(x_max, x2) - max(x_min, x1)
                new_h = min(y_max, y2) - max(y_min, y1)
                
                # Normalize to segment dimensions
                new_x_center /= (x_max - x_min)
                new_y_center /= (y_max - y_min)
                new_w /= (x_max - x_min)
                new_h /= (y_max - y_min)
                
                # Only include if the center is within the segment
                if 0 <= new_x_center <= 1 and 0 <= new_y_center <= 1:
                    segment_annotations.append((class_id, new_x_center, new_y_center, new_w, new_h))

        # Save segment image
        segment_path = os.path.join(output_dir, f"{image_name}_{name}.jpg")
        cv2.imwrite(segment_path, segment)
        print(f"Saved {segment_path} with {len(segment_annotations)} heads")

        # Save segment annotations
        if segment_annotations:
            annotation_path = os.path.join(output_dir, f"{image_name}_{name}.txt")
            with open(annotation_path, 'w') as f:
                for class_id, x_center, y_center, w, h in segment_annotations:
                    f.write(f"{int(class_id)} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")
            print(f"Saved annotations to {annotation_path}")
        else:
            print(f"No heads in {segment_path}, skipping annotation file")

# Step 1: Segment all images
print("Starting segmentation of full images...")
for image_path, annotation_path in image_annotation_pairs:
    print(f"\nProcessing {image_path}")
    segment_image(image_path, annotation_path, segmented_dir)
print("\nSegmentation of all images complete!")

# Step 2: Organize the dataset
print("\nOrganizing dataset...")

# Get all segmented images and annotations
image_files = [f for f in os.listdir(segmented_dir) if f.endswith('.jpg')]
annotation_files = [f for f in os.listdir(segmented_dir) if f.endswith('.txt')]

# Ensure we have matching pairs
image_files.sort()
annotation_files.sort()
print(f"Found {len(image_files)} images and {len(annotation_files)} annotations.")

# Pair images with annotations
pairs = [(img, img.replace('.jpg', '.txt')) for img in image_files]

# Shuffle and split into train (80%) and val (20%)
random.shuffle(pairs)
train_split = int(0.8 * len(pairs))  # 80% for training
train_pairs = pairs[:train_split]    # ~64 images
val_pairs = pairs[train_split:]      # ~16 images

# Copy segmented files to train directory
for img_file, txt_file in train_pairs:
    shutil.copy(os.path.join(segmented_dir, img_file), os.path.join(train_images_dir, img_file))
    if os.path.exists(os.path.join(segmented_dir, txt_file)):
        shutil.copy(os.path.join(segmented_dir, txt_file), os.path.join(train_labels_dir, txt_file))
    else:
        print(f"Warning: Annotation {txt_file} not found for {img_file}")

# Copy segmented files to val directory
for img_file, txt_file in val_pairs:
    shutil.copy(os.path.join(segmented_dir, img_file), os.path.join(val_images_dir, img_file))
    if os.path.exists(os.path.join(segmented_dir, txt_file)):
        shutil.copy(os.path.join(segmented_dir, txt_file), os.path.join(val_labels_dir, txt_file))
    else:
        print(f"Warning: Annotation {txt_file} not found for {img_file}")

# Add full images to the training set
full_images_dir = "/Users/deen/Desktop/head count/images/"
full_image_files = [f for f in os.listdir(full_images_dir) if f.endswith('.jpg')]
for img_file in full_image_files:
    txt_file = img_file.replace('.jpg', '.txt')
    shutil.copy(os.path.join(full_images_dir, img_file), os.path.join(train_images_dir, img_file))
    if os.path.exists(os.path.join(full_images_dir, txt_file)):
        shutil.copy(os.path.join(full_images_dir, txt_file), os.path.join(train_labels_dir, txt_file))
    else:
        print(f"Warning: Annotation {txt_file} not found for {img_file}")

# Create data.yaml file
yaml_content = f"""\
train: {os.path.abspath(train_dir)}
val: {os.path.abspath(val_dir)}
nc: 1  # Number of classes
names: ['head']  # Class names
"""

yaml_path = os.path.join(dataset_dir, "data.yaml")
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

# Print summary
print(f"\nDataset organized at: {os.path.abspath(dataset_dir)}")
print(f"Training set: {len(train_pairs) + len(full_image_files)} images (including {len(full_image_files)} full images)")
print(f"Validation set: {len(val_pairs)} images")
print(f"data.yaml created at: {yaml_path}")
print("\nDataset preparation complete!")

# Importing pretrained Yolov8 nano model

In [ ]:
# Import YOLO
from ultralytics import YOLO

# Load the pre-trained YOLOv8 nano model
model = YOLO("yolov8n.pt")
print("Loaded pre-trained YOLOv8n model.")

# Train with the segmented dataset
model.train(
    data="head_count_dataset1/data.yaml",
    epochs=100,
    imgsz=640,
    batch=4,
    lr0=0.001,
    patience=20,
    name="head_count_model_v1",
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,
    flipud=0.5,
    fliplr=0.5,
    mosaic=0.9,
    scale=0.7,
    box=0.1,
    cls=0.7,
    mixup=0.2,
    iou=0.6
)

print("Training complete! Model saved in 'runs/train/head_count_model_v1'.")

# Testing 

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os

# Model path
model_path = "runs/detect/head_count_model_v1/weights/best.pt"

# Verify model exists
if not os.path.exists(model_path):
    print(f"Error: Model file not found at '{model_path}'.")
    raise FileNotFoundError(f"Model file not found at '{model_path}'.")

# Load the custom-trained YOLOv8 model
model = YOLO(model_path)
print(f"Loaded model from: {model_path}")

# Function to count heads
def count_heads(img, confidence_threshold=0.15, is_full_image=False):
    if img is None:
        print("Error: Image not loaded.")
        return 0, [], None
    
    # Lower confidence threshold for full images
    conf_thres = 0.1 if is_full_image else confidence_threshold
    
    results = model(img, imgsz=640, conf=conf_thres, iou=0.5)
    head_count = 0
    confidences = []
    all_scores = []

    for result in results:
        boxes = result.boxes
        print(f"Total boxes detected: {len(boxes)}")
        for box in boxes:
            conf = box.conf.item()
            cls = int(box.cls)
            all_scores.append(conf)
            print(f"Box: Class={cls}, Confidence={conf:.2f}")
            if cls == 0:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img, f"Head: {conf:.2f}", (x1, y1-10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                head_count += 1
                confidences.append(conf)

    print(f"All detection scores: {all_scores}")
    return head_count, confidences, img

# Process the image and plot results
def process_image(image_path, is_full_image=False):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not load image at {image_path}")
        return
    
    head_count, confidences, processed_img = count_heads(img, is_full_image=is_full_image)
    
    # Convert BGR to RGB for matplotlib display
    processed_img_rgb = cv2.cvtColor(processed_img, cv2.COLOR_BGR2RGB)
    
    # Display the image with bounding boxes
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(processed_img_rgb)
    plt.title("Head Detection")
    plt.axis("off")
    
    # Plot the number of detected heads
    plt.subplot(1, 2, 2)
    plt.bar(["Image"], [head_count], color='skyblue')
    plt.xlabel("Image")
    plt.ylabel("Number of Detected Heads")
    plt.title("Heads Detected")
    
    plt.tight_layout()
    plt.show()
    
    # Plot confidence distribution
    if confidences:
        plt.figure(figsize=(5, 3))
        plt.hist(confidences, bins=10, color='lightgreen', edgecolor='black')
        plt.xlabel("Confidence Score")
        plt.ylabel("Frequency")
        plt.title("Confidence Distribution")
        plt.show()
    
    print(f"Detected heads: {head_count}")
    if confidences:
        print(f"Average confidence: {np.mean(confidences):.2f}")

# Test on all segmented images
segmented_images = [
    "segmented_images/DSC09724_top_left.jpg",
    "segmented_images/DSC09724_top_right.jpg",
    "segmented_images/DSC09724_bottom_left.jpg",
    "segmented_images/DSC09724_bottom_right.jpg"
]

for seg_path in segmented_images:
    if os.path.exists(seg_path):
        print(f"\nTesting on: {seg_path}")
        process_image(seg_path, is_full_image=False)

# Test on full image
image_path = "/Users/deen/Desktop/Head count 2/images/DSC09724.JPG"
print(f"\nTesting on full image: {image_path}")
process_image(image_path, is_full_image=True)

# adding full images to the train set 

In [ ]:
import os
import shutil

# Paths
full_images_dir = "/Users/deen/Desktop/Head count 2/images/"
train_images_dir = "head_count_dataset1/train/images"
train_labels_dir = "head_count_dataset1/train/labels"

# Verify the full images directory exists
print(f"Full images directory: {full_images_dir}")
if not os.path.exists(full_images_dir):
    raise FileNotFoundError(f"Full images directory not found at: {full_images_dir}")

# Create directories if they don't exist
os.makedirs(train_images_dir, exist_ok=True)
os.makedirs(train_labels_dir, exist_ok=True)

# List of full images (handle .jpg and .JPG extensions)
full_image_files = [f for f in os.listdir(full_images_dir) if f.lower().endswith(('.jpg', '.jpeg'))]
print(f"Found {len(full_image_files)} full images: {full_image_files}")

# Copy full images and annotations to train directory
for img_file in full_image_files:
    img_path = os.path.join(full_images_dir, img_file)
    # Match the annotation file based on the image filename (case-insensitive for extension)
    base_name = os.path.splitext(img_file)[0]
    possible_txt_extensions = [f"{base_name}.txt", f"{base_name}.TXT"]
    txt_file = None
    for ext in possible_txt_extensions:
        if os.path.exists(os.path.join(full_images_dir, ext)):
            txt_file = ext
            break
    
    txt_path = os.path.join(full_images_dir, txt_file) if txt_file else None
    
    # Copy image
    shutil.copy(img_path, os.path.join(train_images_dir, img_file))
    print(f"Copied image: {img_file}")
    
    # Copy annotation if it exists
    if txt_path and os.path.exists(txt_path):
        shutil.copy(txt_path, os.path.join(train_labels_dir, txt_file))
        print(f"Copied annotation: {txt_file}")
    else:
        print(f"Warning: Annotation not found for {img_file}")

# Verify the updated dataset
train_images = len([f for f in os.listdir(train_images_dir) if f.lower().endswith(('.jpg', '.jpeg'))])
val_images = len([f for f in os.listdir("head_count_dataset1/val/images") if f.lower().endswith(('.jpg', '.jpeg'))])
print(f"\nUpdated dataset:")
print(f"Training set: {train_images} images (including {len(full_image_files)} full images)")
print(f"Validation set: {val_images} images")

# re training (model_v2)

In [ ]:
from ultralytics import YOLO

# Load the current model
model = YOLO("runs/detect/head_count_model_v1/weights/best.pt")
print("Loaded head_count_model_v1.")

# Train with the updated dataset (now includes full images)
model.train(
    data="head_count_dataset1/data.yaml",
    epochs=100,
    imgsz=640,
    batch=4,
    lr0=0.001,
    patience=20,
    name="head_count_model_v2",
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,
    flipud=0.5,
    fliplr=0.5,
    mosaic=0.9,
    scale=0.7,
    box=0.1,
    cls=0.7,
    mixup=0.2,
    iou=0.6
)

print("Training complete! Model saved in 'runs/train/head_count_model_v2'.")

# re testing with model_v2

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os

# Update model path to head_count_model_v2
model_path = "runs/detect/head_count_model_v2/weights/best.pt"

# Verify model exists
if not os.path.exists(model_path):
    print(f"Error: Model file not found at '{model_path}'.")
    raise FileNotFoundError(f"Model file not found at '{model_path}'.")

# Load the custom-trained YOLOv8 model
model = YOLO(model_path)
print(f"Loaded model from: {model_path}")

# Function to count heads
def count_heads(img, confidence_threshold=0.15, is_full_image=False):
    if img is None:
        print("Error: Image not loaded.")
        return 0, [], None
    
    conf_thres = 0.1 if is_full_image else confidence_threshold
    
    results = model(img, imgsz=640, conf=conf_thres, iou=0.5)
    head_count = 0
    confidences = []
    all_scores = []

    for result in results:
        boxes = result.boxes
        print(f"Total boxes detected: {len(boxes)}")
        for box in boxes:
            conf = box.conf.item()
            cls = int(box.cls)
            all_scores.append(conf)
            print(f"Box: Class={cls}, Confidence={conf:.2f}")
            if cls == 0:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img, f"Head: {conf:.2f}", (x1, y1-10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                head_count += 1
                confidences.append(conf)

    print(f"All detection scores: {all_scores}")
    return head_count, confidences, img

# Process the image and plot results
def process_image(image_path, is_full_image=False):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not load image at {image_path}")
        return
    
    head_count, confidences, processed_img = count_heads(img, is_full_image=is_full_image)
    
    processed_img_rgb = cv2.cvtColor(processed_img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(processed_img_rgb)
    plt.title("Head Detection")
    plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.bar(["Image"], [head_count], color='skyblue')
    plt.xlabel("Image")
    plt.ylabel("Number of Detected Heads")
    plt.title("Heads Detected")
    
    plt.tight_layout()
    plt.show()
    
    if confidences:
        plt.figure(figsize=(5, 3))
        plt.hist(confidences, bins=10, color='lightgreen', edgecolor='black')
        plt.xlabel("Confidence Score")
        plt.ylabel("Frequency")
        plt.title("Confidence Distribution")
        plt.show()
    
    print(f"Detected heads: {head_count}")
    if confidences:
        print(f"Average confidence: {np.mean(confidences):.2f}")

# Test on all segmented images
segmented_images = [
    "segmented_images/DSC09724_top_left.jpg",
    "segmented_images/DSC09724_top_right.jpg",
    "segmented_images/DSC09724_bottom_left.jpg",
    "segmented_images/DSC09724_bottom_right.jpg"
]

for seg_path in segmented_images:
    if os.path.exists(seg_path):
        print(f"\nTesting on: {seg_path}")
        process_image(seg_path, is_full_image=False)

# Test on full image
image_path = "/Users/deen/Desktop/Head count 2/images/DSC09724.JPG"
print(f"\nTesting on full image: {image_path}")
process_image(image_path, is_full_image=True)

# training v3 


In [ ]:
from ultralytics import YOLO

# Load the current model
model = YOLO("runs/detect/head_count_model_v2/weights/best.pt")
print("Loaded head_count_model_v2.")

# Train with the updated dataset (now includes full images)
model.train(
    data="head_count_dataset1/data.yaml",
    epochs=100,
    imgsz=640,
    batch=4,
    lr0=0.001,
    patience=20,
    name="head_count_model_v3",
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,
    flipud=0.5,
    fliplr=0.5,
    mosaic=0.9,
    scale=0.7,
    box=0.15,  # Increased to reduce false positives
    cls=0.8,   # Increased to reduce false positives
    mixup=0.2,
    iou=0.6
)

print("Training complete! Model saved in 'runs/train/head_count_model_v3'.")

# final Testing with model_V3 

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os

# Model path for head_count_model_v3
model_path = "runs/detect/head_count_model_v3/weights/best.pt"

# Verify model exists
if not os.path.exists(model_path):
    print(f"Error: Model file not found at '{model_path}'.")
    raise FileNotFoundError(f"Model file not found at '{model_path}'.")

# Load the custom-trained YOLOv8 model
model = YOLO(model_path)
print(f"Loaded model from: {model_path}")

# Function to count heads
def count_heads(img, confidence_threshold=0.19, is_full_image=False):
    if img is None:
        print("Error: Image not loaded.")
        return 0, [], None
    
    conf_thres = 0.1 if is_full_image else confidence_threshold
    
    results = model(img, imgsz=640, conf=conf_thres, iou=0.5)
    head_count = 0
    confidences = []
    all_scores = []

    for result in results:
        boxes = result.boxes
        print(f"Total boxes detected: {len(boxes)}")
        for box in boxes:
            conf = box.conf.item()
            cls = int(box.cls)
            all_scores.append(conf)
            print(f"Box: Class={cls}, Confidence={conf:.2f}")
            if cls == 0:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img, f"Head: {conf:.2f}", (x1, y1-10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                head_count += 1
                confidences.append(conf)

    print(f"All detection scores: {all_scores}")
    return head_count, confidences, img

# Process the image and plot results
def process_image(image_path, is_full_image=False):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not load image at {image_path}")
        return
    
    head_count, confidences, processed_img = count_heads(img, is_full_image=is_full_image)
    
    processed_img_rgb = cv2.cvtColor(processed_img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(processed_img_rgb)
    plt.title("Head Detection")
    plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.bar(["Image"], [head_count], color='skyblue')
    plt.xlabel("Image")
    plt.ylabel("Number of Detected Heads")
    plt.title("Heads Detected")
    
    plt.tight_layout()
    plt.show()
    
    if confidences:
        plt.figure(figsize=(5, 3))
        plt.hist(confidences, bins=10, color='lightgreen', edgecolor='black')
        plt.xlabel("Confidence Score")
        plt.ylabel("Frequency")
        plt.title("Confidence Distribution")
        plt.show()
    
    print(f"Detected heads: {head_count}")
    if confidences:
        print(f"Average confidence: {np.mean(confidences):.2f}")

# Test on all segmented images
segmented_images = [
    "segmented_images/DSC09724_top_left.jpg", #replace with any segmented image
    "segmented_images/DSC09724_top_right.jpg",
    "segmented_images/DSC09724_bottom_left.jpg",
    "segmented_images/DSC09724_bottom_right.jpg"
]

for seg_path in segmented_images:
    if os.path.exists(seg_path):
        print(f"\nTesting on: {seg_path}")
        process_image(seg_path, is_full_image=False)

# Test on full image
image_path = "/Users/deen/Desktop/Head count 2/images/DSC09724.JPG" #replace with your image path
print(f"\nTesting on full image: {image_path}")
process_image(image_path, is_full_image=True)